# Reject a status field without a JSON representation

Export a state whose custom field cannot be serialized.


## Declare the model

Run these definitions first. This notebook is independent of the other examples.


In [1]:
from pydantic import ConfigDict, Field
from pydantic_core import PydanticSerializationError
from aoa.action_machine.domain import BaseEntity, BaseStateEntity, Lifecycle, StateVocabulary
from aoa.action_machine.graph.nodes.lifecycle_graph_node import LifeCycleGraphNode


class Opaque:
    """Provide no JSON representation."""


class OrderStateEntity(BaseStateEntity):
    """Allow a Python object that has no JSON form."""
    model_config = ConfigDict(arbitrary_types_allowed=True)
    value: Opaque = Field(description="An opaque Python value")


class OrderLifecycle(Lifecycle, StateVocabulary):
    """Use otherwise valid transitions."""
    DRAFT = OrderStateEntity(code="draft", display_name="Draft", value=Opaque())
    PAID = OrderStateEntity(code="paid", display_name="Paid", value=Opaque())
    _template = Lifecycle().state(DRAFT).to(PAID).initial().state(PAID).final()


class OrderEntity(BaseEntity):
    """Host the lifecycle used for graph export."""
    lifecycle: OrderLifecycle = Field(description="Current order state")


## Run the case


In [2]:
try:
    LifeCycleGraphNode(OrderEntity, "lifecycle", OrderLifecycle)
except PydanticSerializationError as error:
    print(type(error).__name__ + ": " + str(error))
else:
    raise AssertionError("The invalid declaration was accepted")


PydanticSerializationError: Unable to serialize unknown type: <class '__main__.Opaque'>


## What the result means

arbitrary_types_allowed permits construction of the Python model, but does not supply JSON encoding. Export fails rather than silently dropping the field. Use JSON-compatible domain values or explicitly exclude a field that should not be exported.
